###Transform Constructors Data
1. Read bronze constructors table
2. Keep only the columns required for analytics (Drop url column)
3. Standardise column names using snake case
4. Rename columns to make more meanimgful
5. Remove duplicate records
6. Transform values of column nationality to title case
7. Write the transformed data to constructors table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/03.silver_helpers

In [0]:
from pyspark.sql import functions as f

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema }.constructors"

In [0]:
constructors_df = spark.table(bronze_table).filter(f.col("batch_id") == v_batch_id)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
    .withColumnRenamed("constructorId", "constructor_id")
)

In [0]:
#display(constructors_renamed_df)

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
#display(constructors_distinct_df)

In [0]:
constructors_final_df = constructors_distinct_df\
    .withColumn("nationality", f.initcap(f.col("nationality")))

In [0]:
write_to_silver(
    constructors_final_df,
    silver_table,
    "t.constructor_Id = s.constructor_id",
    [
        "name",
        "nationality",
        "current_timestamp",
        "source_file",
        "batch_id"
    ]
)

In [0]:
#display(spark.table(silver_table))

In [0]:
%sql
--select * from formula1.silver.constructors